In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
!unzip -q /content/drive/MyDrive/multiobject.zip -d /content/data
!pip install -q ultralytics
!find /content/data -maxdepth 3 -type d

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 9.7 MB/s eta 0:00:00
/content/data
/content/data/multiobject
/content/data/multiobject/labels
/content/data/multiobject/labels/val
/content/data/multiobject/labels/test
/content/data/multiobject/labels/train
/content/data/multiobject/images
/content/data/multiobject/images/val
/content/data/multiobject/images/test
/content/data/multiobject/images/train


In [4]:
import re
p = "/content/data/multiobject/data.yaml"
t = open(p).read()
t = re.sub(r"(?m)^path:.*$", "path: /content/data/multiobject", t)
open(p, "w").write(t)
print(t[:200])


path: /content/data/multiobject
train: images/train
val: images/val
test: images/test
nc: 73
names:
  0: OBJ001
  1: OBJ002
  2: OBJ003
  3: OBJ004
  4: OBJ005
  5: OBJ006
  6: OBJ007
  7: OBJ008
  8:


In [5]:
from ultralytics import YOLO
model = YOLO("yolov8s.pt")
model.train(
    data="/content/data/multiobject/data.yaml",
    epochs=50, imgsz=640, batch=16, seed=42,
    project="/content/drive/MyDrive/yolo_runs", name="yolov8s",
    patience=15, fliplr=0.5, flipud=0.0, degrees=0.0, close_mosaic=10,
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/multiobject/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79d32fb533f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0

In [6]:
import os, csv
from ultralytics import YOLO
W = "/content/drive/MyDrive/yolo_runs/yolov8s/weights/best.pt"
D = "/content/data/multiobject/data.yaml"
model = YOLO(W)
m = model.val(data=D, split="test", conf=0.001, plots=True,
              project="/content/drive/MyDrive/yolo_runs", name="yolov8s_test")
print(f"TEST mAP50 {m.box.map50:.4f} | mAP50-95 {m.box.map:.4f} | P {m.box.mp:.4f} | R {m.box.mr:.4f}")

# per-class AP
rows = sorted(zip(m.box.ap_class_index, m.box.ap50), key=lambda r: r[1])
print("Lowest 5 classes (AP50):", [(model.names[int(k)], round(float(v), 3)) for k, v in rows[:5]])

# image-level: does the predicted set of object IDs match the true set?
img_dir, lbl_dir = "/content/data/multiobject/images/test", "/content/data/multiobject/labels/test"
exact = tp = fp = fn = n = 0
for f in sorted(os.listdir(img_dir)):
    gt = {int(l.split()[0]) for l in open(f"{lbl_dir}/{os.path.splitext(f)[0]}.txt") if l.strip()}
    pr = {int(c) for c in model.predict(f"{img_dir}/{f}", conf=0.25, verbose=False)[0].boxes.cls.tolist()}
    exact += pr == gt; tp += len(pr & gt); fp += len(pr - gt); fn += len(gt - pr); n += 1
print(f"Image-level: exact set match {exact/n:.3f} | ID precision {tp/max(tp+fp,1):.3f} | ID recall {tp/max(tp+fn,1):.3f}")

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,153,835 parameters, 0 gradients, 28.6 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 6.9±1.9 MB/s, size: 42.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/data/multiobject/labels/test... 300 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 300/300 261.6it/s 1.1s
val: New cache created: /content/data/multiobject/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 1.7it/s 11.3s
                   all        300       1174      0.963       0.91      0.965      0.963
                OBJ001         15         15      0.987          1      0.995      0.995
                OBJ002         21         21          1      0.995      0.995      0.995
   